In [1]:
### === IMPORTS === ###
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display, HTML
# import math
# import matplotlib.pyplot as plt
# import seaborn as sns
# import statsmodels.api as sm
# from sklearn.preprocessing import StandardScaler
# from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.metrics import make_scorer

pd.set_option('display.max_columns', None)

# Data Preprocessing

In [2]:
### === DATA LODA === ###
# ----------------------------------
# Statsbomb Event Data Retrieval
# ----------------------------------
EDA_DIR = Path.cwd()  # should be .../<parent folder name>/eda
PROJECT_ROOT = EDA_DIR.parent # .../<parent folder name>
STATSBOMB_DIR = PROJECT_ROOT / "data" / "Statsbomb"

EVENTS_PATH = STATSBOMB_DIR / "events.parquet"
MATCHES_PATH = STATSBOMB_DIR / "matches.parquet"

# Read full parquet files
events_df = pd.read_parquet(EVENTS_PATH)
matches_df = pd.read_parquet(MATCHES_PATH)

print("events_df shape :", events_df.shape)
print("matches_df shape:", matches_df.shape)

events_df shape : (12188949, 112)
matches_df shape: (3464, 36)


In [3]:
# ------------------------------------------------------------
# 2. Keep only regulation-time events (periods 1 and 2)
# ------------------------------------------------------------
events_df = events_df.loc[events_df["period"].isin([1, 2])].copy()

print("Regulation-only events_df shape:", events_df.shape)

Regulation-only events_df shape: (12134916, 112)


In [4]:
# ------------------------------------------------------------
# 2b. Keep only mens games occurring in year 2000 or later 
# ------------------------------------------------------------
matches_df = pd.read_parquet(MATCHES_PATH)

# Parse date and extract year
matches_df["match_date"] = pd.to_datetime(matches_df["match_date"], errors="coerce")
matches_df["year"] = matches_df["match_date"].dt.year

matches_df = matches_df[
    (matches_df["year"] >= 2000) 
    & (matches_df["gender"] == "male")
    & (matches_df["is_youth"] == False)
]

print("Unique matches:", matches_df.shape[0])
display(matches_df.head(2))

Unique matches: 2889


,match_id,match_date,match_week,match_status,match_status_360,kickoff,home_score,away_score,competition_id,competition,competition_stage,season_id,season,home_team_id,home_team,home_managers,away_team_id,away_team,away_managers,stadium_id,stadium,referee_id,referee,last_updated,last_updated_360,data_version,shot_fidelity_version,xy_fidelity_version,competition_name,gender,is_youth,is_international,country_name,season_name,match_updated,match_available_360,year
0,9880,2018-04-14,32,available,scheduled,16:15:00,2,1,11,La Liga,Regular Season,1,2017/2018,217,Barcelona,"[{""id"":227,""name"":""Ernesto Valverde Tejedor"",""...",207,Valencia,"[{""id"":211,""name"":""Marcelino García Toral"",""ni...",342.0,Spotify Camp Nou,2728.0,Carlos del Cerro Grande,2023-02-08T17:23:53.901920,2021-06-13T16:17:31.694,1.1.0,2,2,La Liga,male,False,False,Spain,2017/2018,2025-07-14T10:01:16.674864,None,2018
1,9912,2018-04-29,35,available,scheduled,20:45:00,2,4,11,La Liga,Regular Season,1,2017/2018,219,RC Deportivo La Coruña,"[{""id"":371,""name"":""Clarence Seedorf"",""nickname...",217,Barcelona,"[{""id"":227,""name"":""Ernesto Valverde Tejedor"",""...",4658.0,Estadio Abanca-Riazor,2602.0,Ricardo De Burgos Bengoetxea,2022-12-05T14:42:44.641092,2021-06-13T16:17:31.694,1.1.0,2,2,La Liga,male,False,False,Spain,2017/2018,2025-07-14T10:01:16.674864,None,2018


In [5]:
# ------------------------------------------------------------
# 5. Join assigned cutoff back to event-level data
# ------------------------------------------------------------
events_df = events_df.merge(
    matches_df[["match_id"]],
    on="match_id",
    how="inner",
    validate="many_to_one"
)

print("Events with cutoff columns shape:", events_df.shape)

Events with cutoff columns shape: (10215925, 112)


In [6]:
def create_match_results_matrix(e, m):
    """
    Build a match-team feature table from a filtered events dataframe.
    
    Parameters
    ----------
    e : pandas.DataFrame
        Filtered event-level dataframe. Must contain:
        ['match_id', 'team', 'type', 'year', ...]
        
    m : pandas.DataFrame
        Match results spine with columns:
        ['match_id', 'team', 'goals_scored', 'result']
    
    Returns
    -------
    pandas.DataFrame
        One row per match_id x team with aggregated features,
        opponent totals, differentials, year, and binary win column.
    """
    # ------------------------------------------------------------
    # 1. Extract year from events
    # ------------------------------------------------------------
    # year_df = m[["match_id", "year"]]

    # ------------------------------------------------------------
    # 2. Core event aggregations
    # ------------------------------------------------------------
    team_vol = (
        e.groupby(["match_id", "team"], as_index=False)
         .agg(
             n_carries=("type", lambda x: (x == "Carry").sum()),
             n_dribbles=("type", lambda x: (x == "Dribble").sum()),
             n_passes=("type", lambda x: (x == "Pass").sum()),
         )
    )

    # goals scored
    team_goals = (
        e[(e["type"] == "Shot") & (e["shot_outcome"] == "Goal")]
        .groupby(["match_id", "team"])
        .size()
        .reset_index(name="goals_scored_timebucket")
    )
    

    # completed passes
    pass_complete = (
        e.loc[e["type"] == "Pass"]
         .assign(pass_complete=lambda df: df["pass_outcome"].isna().astype(int))
         .groupby(["match_id", "team"], as_index=False)["pass_complete"]
         .sum()
         .rename(columns={"pass_complete": "n_passes_complete"})
    )
    team_vol = team_vol.merge(pass_complete, on=["match_id", "team"], how="left")
    team_vol["n_passes_complete"] = team_vol["n_passes_complete"].fillna(0).astype(int)

    # shots
    team_shots = (
        e.loc[e["type"] == "Shot"]
         .groupby(["match_id", "team"], as_index=False)
         .size()
         .rename(columns={"size": "shots"})
    )

    # xG
    team_xg = (
        e.loc[e["type"] == "Shot"]
         .groupby(["match_id", "team"], as_index=False)["shot_statsbomb_xg"]
         .sum()
         .rename(columns={"shot_statsbomb_xg": "xg_for"})
    )

    # shots on target
    sot_outcomes = ["Goal", "Saved", "Saved to Post", "Saved Off Target"]
    team_sot = (
        e.loc[(e["type"] == "Shot") & (e["shot_outcome"].isin(sot_outcomes))]
         .groupby(["match_id", "team"], as_index=False)
         .size()
         .rename(columns={"size": "shots_on_target"})
    )

    # pressures
    team_press = (
        e.loc[e["type"] == "Pressure"]
         .groupby(["match_id", "team"], as_index=False)
         .size()
         .rename(columns={"size": "pressures"})
    )

    # final third passes
    team_ft_pass = (
        e.loc[(e["type"] == "Pass") & (e["pass_end_location_x"] > 80)]
         .groupby(["match_id", "team"], as_index=False)
         .size()
         .rename(columns={"size": "final_third_passes"})
    )

    # final third carries
    team_ft_carry = (
        e.loc[(e["type"] == "Carry") & (e["carry_end_location_x"] > 80)]
         .groupby(["match_id", "team"], as_index=False)
         .size()
         .rename(columns={"size": "final_third_carries"})
    )
    
    # ------------------------------------------------------------
    # 3. Start from match-results spine and join all features
    # ------------------------------------------------------------
    df = (
        m.copy()
         # .merge(year_df, on="match_id", how="left")
         .merge(team_shots, on=["match_id", "team"], how="left")
         .merge(team_xg, on=["match_id", "team"], how="left")
         .merge(team_sot, on=["match_id", "team"], how="left")
         .merge(team_press, on=["match_id", "team"], how="left")
         .merge(team_ft_pass, on=["match_id", "team"], how="left")
         .merge(team_ft_carry, on=["match_id", "team"], how="left")
         .merge(team_vol, on=["match_id", "team"], how="left")
         .merge(team_goals, on=["match_id", "team"], how="left")
    )

    # fill nulls
    fill_zero_int = [
        "shots", "shots_on_target", "pressures",
        "final_third_passes", "final_third_carries",
        "n_carries", "n_dribbles", "n_passes", "n_passes_complete", "goals_scored_timebucket"
    ]
    for col in fill_zero_int:
        df[col] = df[col].fillna(0).astype(int)
    
    df["xg_for"] = df["xg_for"].fillna(0.0)
    df["final_third_entries"] = df["final_third_passes"] + df["final_third_carries"]
    
    # ------------------------------------------------------------
    # 4. Opponent totals / against columns
    # ------------------------------------------------------------
    # full game columns
    df["goals_scored"] = df["goals_scored"].astype(int)
    df["match_goals_total"] = df.groupby("match_id")["goals_scored"].transform("sum")
    df["goals_conceded"] = df["match_goals_total"] - df["goals_scored"]

    # event derived / timebucket totals
    df["goals_scored_timebucket"] = df["goals_scored_timebucket"].astype(int)
    df["match_goals_total_timebucket"] = df.groupby("match_id")["goals_scored_timebucket"].transform("sum")
    df["match_shots_total"] = df.groupby("match_id")["shots"].transform("sum")
    df["match_sot_total"] = df.groupby("match_id")["shots_on_target"].transform("sum")
    df["match_xg_total"] = df.groupby("match_id")["xg_for"].transform("sum")
    df["match_ft_total"] = df.groupby("match_id")["final_third_entries"].transform("sum")
    df["match_press_total"] = df.groupby("match_id")["pressures"].transform("sum")

    # event derived / timebucket against
    df["shots_against"] = df["match_shots_total"] - df["shots"]
    df["sot_against"] = df["match_sot_total"] - df["shots_on_target"]
    df["xg_against"] = df["match_xg_total"] - df["xg_for"]
    df["final_third_against"] = df["match_ft_total"] - df["final_third_entries"]
    df["pressure_against"] = df["match_press_total"] - df["pressures"]
    df["goals_conceded_timebucket"] = df["match_goals_total_timebucket"] - df["goals_scored_timebucket"]

    # ------------------------------------------------------------
    # 5. Differential features
    # ------------------------------------------------------------
    df["goal_diff"] = df["goals_scored"] - df["goals_conceded"]
    df["goal_diff_timebucket"] = df["goals_scored_timebucket"] - df["goals_conceded_timebucket"]
    df["shot_diff"] = df["shots"] - df["shots_against"]
    df["sot_diff"] = df["shots_on_target"] - df["sot_against"]
    df["xg_diff"] = df["xg_for"] - df["xg_against"]
    df["final_third_diff"] = df["final_third_entries"] - df["final_third_against"]
    df["pressure_diff"] = df["pressures"] - df["pressure_against"]

    # ------------------------------------------------------------
    # 6. Passing / carrying / dribbling against + differentials
    # ------------------------------------------------------------
    df["match_total_carries"] = df.groupby("match_id")["n_carries"].transform("sum")
    df["match_total_dribbles"] = df.groupby("match_id")["n_dribbles"].transform("sum")
    df["match_total_passes"] = df.groupby("match_id")["n_passes"].transform("sum")
    df["match_total_passes_complete"] = df.groupby("match_id")["n_passes_complete"].transform("sum")

    df["carries_against"] = df["match_total_carries"] - df["n_carries"]
    df["dribbles_against"] = df["match_total_dribbles"] - df["n_dribbles"]
    df["passes_against"] = df["match_total_passes"] - df["n_passes"]
    df["passes_complete_against"] = df["match_total_passes_complete"] - df["n_passes_complete"]

    df["carry_diff"] = df["n_carries"] - df["carries_against"]
    df["dribble_diff"] = df["n_dribbles"] - df["dribbles_against"]
    df["pass_diff"] = df["n_passes"] - df["passes_against"]
    df["pass_complete_diff"] = df["n_passes_complete"] - df["passes_complete_against"]

    # pass completion rate
    df["pass_completion_rate"] = np.where(
        df["n_passes"] > 0,
        df["n_passes_complete"] / df["n_passes"],
        np.nan
    )

    # ------------------------------------------------------------
    # 7. Binary outcome target
    # ------------------------------------------------------------
    df["win"] = np.where(
        df["result"] == "Win", 1,
        np.where(df["result"] == "Loss", 0, np.nan)
    )


    # ------------------------------------------------------------
    # 8. Drop temporary total columns
    # ------------------------------------------------------------
    drop_cols = [
        "match_goals_total", "match_goals_total_timebucket",
        "match_shots_total", "match_sot_total",
        "match_xg_total", "match_ft_total", "match_press_total",
        "match_total_carries", "match_total_dribbles",
        "match_total_passes", "match_total_passes_complete"
    ]
    df = df.drop(columns=drop_cols)
    
    return df

In [7]:
# ------------------------------------------------------------
# Build match_results table from matches_df
# Output: one row per match_id x team
# ------------------------------------------------------------
def build_match_results(matches_df):
    m = matches_df.copy()

    # parse date and extract year
    m["match_date"] = pd.to_datetime(m["match_date"], errors="coerce")
    m["year"] = m["match_date"].dt.year

    # -------------------------
    # home team rows
    # -------------------------
    home_df = m[[
        "match_id",
        "match_date",
        "year",
        "competition_name",
        "season_name",
        "gender",
        "is_youth",
        "is_international",
        "home_team",
        "away_team",
        "home_score",
        "away_score"
    ]].copy()

    home_df = home_df.rename(columns={
        "home_team": "team",
        "away_team": "opponent",
        "home_score": "goals_scored",
        "away_score": "goals_conceded"
    })

    # -------------------------
    # away team rows
    # -------------------------
    away_df = m[[
        "match_id",
        "match_date",
        "year",
        "competition_name",
        "season_name",
        "gender",
        "is_youth",
        "is_international",
        "home_team",
        "away_team",
        "home_score",
        "away_score"
    ]].copy()

    away_df = away_df.rename(columns={
        "away_team": "team",
        "home_team": "opponent",
        "away_score": "goals_scored",
        "home_score": "goals_conceded"
    })

    # combine home + away into long format
    match_results = pd.concat([home_df, away_df], ignore_index=True)

    # create result label
    match_results["result"] = np.where(
        match_results["goals_scored"] > match_results["goals_conceded"], "Win",
        np.where(
            match_results["goals_scored"] < match_results["goals_conceded"], "Loss", "Draw"
        )
    )

    # optional binary target for later modeling
    match_results["win"] = np.nan
    match_results.loc[match_results["result"] == "Win", "win"] = 1
    match_results.loc[match_results["result"] == "Loss", "win"] = 0

    # reorder columns
    match_results = match_results[[
        "match_id",
        "match_date",
        "year",
        "competition_name",
        "season_name",
        "gender",
        "is_youth",
        "is_international",
        "team",
        "opponent",
        "goals_scored",
        "goals_conceded",
        "result",
        "win"
    ]].sort_values(["match_id", "team"]).reset_index(drop=True)

    return match_results

In [8]:
# matches_df = pd.read_parquet(MATCHES_PATH)
match_results = build_match_results(matches_df)
match_results = match_results[["match_id", "team", "opponent", "year", "result", "win", "goals_scored"]]

print(match_results.shape)
display(match_results.head())

print(match_results.groupby("match_id").size().value_counts().sort_index())
print(match_results["result"].value_counts(dropna=False))
print(match_results["win"].value_counts(dropna=False))

(5778, 7)


,match_id,team,opponent,year,result,win,goals_scored
0,7525,Russia,Saudi Arabia,2018,Win,1.0,5
1,7525,Saudi Arabia,Russia,2018,Loss,0.0,0
2,7529,Croatia,Nigeria,2018,Win,1.0,2
3,7529,Nigeria,Croatia,2018,Loss,0.0,0
4,7530,Australia,France,2018,Loss,0.0,1


2    2889
Name: count, dtype: int64
result
Win     2183
Loss    2183
Draw    1412
Name: count, dtype: int64
win
1.0    2183
0.0    2183
NaN    1412
Name: count, dtype: int64


In [9]:
bucket_results = []

for bucket in [15, 30, 45, 60, 75, 90]:
    print(f'bucket {bucket}...')
    
    # --------------------------------------------
    # Filter events to everything observed up to bucket
    # --------------------------------------------
    mask_keep = (
        # first-half events up to the bucket
        ((events_df["period"] == 1) & (events_df["minute"] <= bucket)) |
        
        # keep first-half stoppage time for any second-half bucket
        ((events_df["period"] == 1) & (bucket > 45) & (events_df["minute"] <= 60)) |
        
        # second-half events up to the bucket
        ((events_df["period"] == 2) & (bucket > 45) & (events_df["minute"] <= bucket)) |
        
        # keep second-half stoppage time when bucket is full regulation
        ((events_df["period"] == 2) & (bucket == 90))
    )
    
    events_cut = events_df.loc[mask_keep].copy()
    
    # --------------------------------------------
    # Aggregate to match-team feature matrix
    # --------------------------------------------
    df_bucket = create_match_results_matrix(events_cut, match_results)
    df_bucket["total_minutes"] = bucket
    
    bucket_results.append(df_bucket)

df_all_buckets = pd.concat(bucket_results, ignore_index=True)
print("All buckets done!")

bucket 15...
bucket 30...
bucket 45...
bucket 60...
bucket 75...
bucket 90...
All buckets done!


In [11]:
df_all_buckets[(df_all_buckets['match_id']==7525) & ((df_all_buckets['total_minutes']==15) | (df_all_buckets['total_minutes']==45) | (df_all_buckets['total_minutes']==90))]

,match_id,team,opponent,year,result,win,goals_scored,shots,xg_for,shots_on_target,pressures,final_third_passes,final_third_carries,n_carries,n_dribbles,n_passes,n_passes_complete,goals_scored_timebucket,final_third_entries,goals_conceded,shots_against,sot_against,xg_against,final_third_against,pressure_against,goals_conceded_timebucket,goal_diff,goal_diff_timebucket,shot_diff,sot_diff,xg_diff,final_third_diff,pressure_diff,carries_against,dribbles_against,passes_against,passes_complete_against,carry_diff,dribble_diff,pass_diff,pass_complete_diff,pass_completion_rate,total_minutes
0,7525,Russia,Saudi Arabia,2018,Win,1.0,5,4,0.295044,1,29,41,19,51,1,86,58,1,60,0,1,0,0.011307,70,32,0,5,1,3,1,0.283737,-10,-3,61,4,78,57,-10,-3,8,1,0.674419,15
1,7525,Saudi Arabia,Russia,2018,Loss,0.0,0,1,0.011307,0,32,38,32,61,4,78,57,0,70,5,4,1,0.295044,60,29,1,-5,-1,-3,-1,-0.283737,10,3,51,1,86,58,10,3,-8,-1,0.730769,15
11556,7525,Russia,Saudi Arabia,2018,Win,1.0,5,5,0.544284,2,66,76,50,117,6,181,117,2,126,0,4,0,0.197827,135,74,0,5,2,1,2,0.346457,-9,-8,220,5,287,225,-103,1,-106,-108,0.646409,45
11557,7525,Saudi Arabia,Russia,2018,Loss,0.0,0,4,0.197827,0,74,75,60,220,5,287,225,0,135,5,5,2,0.544284,126,66,2,-5,-2,-1,-2,-0.346457,9,8,117,6,181,117,103,-1,106,108,0.783972,45
28890,7525,Russia,Saudi Arabia,2018,Win,1.0,5,14,1.281601,7,147,148,105,269,12,388,272,5,253,0,6,0,0.248806,209,121,0,5,5,8,7,1.032794,44,26,440,11,572,458,-171,1,-184,-186,0.701031,90
28891,7525,Saudi Arabia,Russia,2018,Loss,0.0,0,6,0.248807,0,121,122,87,440,11,572,458,0,209,5,14,7,1.281601,253,147,5,-5,-5,-8,-7,-1.032794,-44,-26,269,12,388,272,171,-1,184,186,0.800699,90


In [14]:
df_all_buckets.shape

(34668, 43)

In [16]:
col_order = [
    # identifiers
    "match_id", "team", "opponent", "year", "total_minutes",

    # targets (full game)
    "result", "win", "goals_scored", "goals_conceded", "goal_diff",

    # timebucket score state
    "goals_scored_timebucket", "goals_conceded_timebucket", "goal_diff_timebucket",

    # attacking (for)
    "shots", "shots_on_target", "xg_for",
    "final_third_entries", "final_third_passes", "final_third_carries",
    "pressures",

    # defending (against)
    "shots_against", "sot_against", "xg_against",
    "final_third_against", "pressure_against",

    # differentials
    "shot_diff", "sot_diff", "xg_diff",
    "final_third_diff", "pressure_diff",

    # possession (for)
    "n_carries", "n_dribbles", "n_passes", "n_passes_complete",
    "pass_completion_rate",

    # possession (against)
    "carries_against", "dribbles_against", "passes_against", "passes_complete_against",

    # possession differentials
    "carry_diff", "dribble_diff", "pass_diff", "pass_complete_diff"
]

df_all_buckets = df_all_buckets[col_order]

In [17]:
df_all_buckets.shape

(34668, 43)

In [18]:
OUTPUT_PATH = "df_all_buckets.csv"

df_all_buckets.to_csv(OUTPUT_PATH, index=False)

print(f"Saved to {OUTPUT_PATH}")

Saved to df_all_buckets.csv
